In [ ]:
import numpy as np
from typing import Union
from pathlib import Path
from skimage.morphology import remove_small_objects
from aicssegmentation.core.pre_processing_utils import (
    intensity_normalization,
    edge_preserving_smoothing_3d,
)
from aicssegmentation.core.vessel import vesselness3D
from aicssegmentation.core.output_utils import (
    save_segmentation,
    generate_segmentation_contour,
)
from scipy.ndimage import zoom
import pandas as pd
from aicsimageio import AICSImage
from aicsimageio.writers import OmeTiffWriter

In [ ]:
def Workflow_actn1(
    struct_img: np.ndarray,
    rescale_ratio: float = -1,
    output_type: str = "default",
    output_path: Union[str, Path] = None,
    fn: Union[str, Path] = None,
    output_func=None,
):
    """
    Classic segmentation workflow wrapper for structure ACTN1 (Alpha-actinin - Actin bundles)

    Parameter:
    -----------
    struct_img: np.ndarray
        the 3D image to be segmented
    rescale_ratio: float
        an optional parameter to allow rescale the image before running the
        segmentation functions, default is no rescaling
    output_type: str
        select how to handle output. Currently, four types are supported:
        1. default: the result will be saved at output_path whose filename is
            original name without extension + "_struct_segmentation.tiff"
        2. array: the segmentation result will be simply returned as a numpy array
        3. array_with_contour: segmentation result will be returned together with
            the contour of the segmentation
        4. customize: pass in an extra output_func to do a special save. All the
            intermediate results, names of these results, the output_path, and the
            original filename (without extension) will be passed in to output_func.
    """
    ##########################################################################
    # PARAMETERS:
    #   note that these parameters are supposed to be fixed for the structure
    #   and work well across different datasets

    intensity_norm_param = [3, 15]
    vesselness_sigma_1 = [2]
    vesselness_cutoff_1 = 0.15
    vesselness_sigma_2 = [1]
    vesselness_cutoff_2 = 0.05
    minArea = 5
    ##########################################################################

    out_img_list = []
    out_name_list = []

    ###################
    # PRE_PROCESSING
    ###################
    # intensity normalization (min/max)
    struct_img = intensity_normalization(struct_img, scaling_param=intensity_norm_param)

    out_img_list.append(struct_img.copy())
    out_name_list.append("im_norm")

    # rescale if needed
    if rescale_ratio > 0:
        struct_img = zoom(struct_img, (1, rescale_ratio, rescale_ratio), order=2)

        struct_img = (struct_img - struct_img.min() + 1e-8) / (struct_img.max() - struct_img.min() + 1e-8)

    # smoothing with edge preserving filter
    structure_img_smooth = edge_preserving_smoothing_3d(struct_img)

    out_img_list.append(structure_img_smooth.copy())
    out_name_list.append("im_smooth")

    ###################
    # core algorithm
    ###################

    # Multi-scale 3D vesselness
    response_1 = vesselness3D(structure_img_smooth, sigmas=vesselness_sigma_1, tau=1, whiteonblack=True)
    response_2 = vesselness3D(structure_img_smooth, sigmas=vesselness_sigma_2, tau=1, whiteonblack=True)

    bw = np.logical_or(response_1 > vesselness_cutoff_1, response_2 > vesselness_cutoff_2)

    ###################
    # POST-PROCESSING
    ###################
    seg = remove_small_objects(bw, min_size=minArea, connectivity=1)

    # output
    seg = seg > 0
    seg = seg.astype(np.uint8)
    seg[seg > 0] = 255

    out_img_list.append(seg.copy())
    out_name_list.append("bw_final")

    if output_type == "default":
        OmeTiffWriter.save(seg, Path(output_path))
    elif output_type == "customize":
        output_func(out_img_list, out_name_list, Path(output_path), fn)
    elif output_type == "array":
        return seg
    elif output_type == "array_with_contour":
        return (seg, generate_segmentation_contour(seg))
    else:
        raise NotImplementedError(f"invalid output type: {output_type}")

In [ ]:
# Configure input file path (CSV with 'raw' column or single image path)
ipfilepath = "/path/to/your/data.csv"
df = pd.read_csv(ipfilepath)

In [ ]:
for imgnameX in df.raw.values:
    img1_reader = AICSImage(imgnameX)
    img1 = img1_reader.data
    img1 = np.squeeze(img1)
    filename = imgnameX.split('/')[-1]
    targetname = filename[:-5] + '_classic_seg.tiff'
    target_path = "/path/to/output/" + targetname
    Workflow_actn1(img1, output_path=target_path)